# Customer Shopping Behaviour: Data Inspection and Preparation

## Notebook Objective

This notebook prepares the dataset for visualization by:

- loading and inspecting the raw data;
- validating its structure and data quality;
- correcting data types;
- removing unnecessary columns;
- creating useful analytical features;
- saving a processed dataset for the visualization notebooks.

# Imports and Display settings

In [38]:
from pathlib import Path

import numpy as np
import pandas as pd

In [39]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

# File path

In [40]:
DATA_PATH = Path("../data/customer_shopping_behavior_Task3.csv")
#Will print the dataset path to verify that the path is correct and the file exists
#print("Dataset path:", DATA_PATH.resolve())

# Data Inspection

### Dataset Load

In [41]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Dataset loaded successfully.
Rows: 10,000
Columns: 26


,Transaction ID,Customer ID,Purchase Date,Age,Gender,Location,Online/Offline,Online Store,Category,Item Purchased,Brand,Color,Size,Quantity,Purchase Amount (₹),Discount (%),Festival/Sale,Shipping Charge (₹),Delivery Speed,Delivery Time (Days),Subscription Status,Payment Method,Review Rating,Return Status,Previous Purchases,Frequency of Purchases
0,TXN500000,CUST001440,2023-07-22,52,Other,Indore,Offline,In-Store Purchase,Clothing,Jeans,Levis,Black,L,1,2215,6,Standard Day,0,N/A (Offline),0,No,Credit Card,5,Not Returned,2,Quarterly
1,TXN500001,CUST002326,2023-01-16,19,Female,Bhubaneswar,Online,Ajio,Clothing,Jacket,Zara,Blue,XS,1,6742,6,Standard Day,0,Express,1,No,Debit Card,4,Returned,3,Quarterly
2,TXN500002,CUST002680,2023-01-24,40,Male,Bangalore,Online,Myntra,Clothing,Saree,Biba,Red,S,1,4672,0,Standard Day,0,Standard,7,Yes,Cash on Delivery,4,Not Returned,13,Monthly
3,TXN500003,CUST001098,2024-09-06,46,Female,Ahmedabad,Offline,In-Store Purchase,Clothing,Jeans,Levis,White,M,1,1539,5,Standard Day,0,N/A (Offline),0,No,UPI,3,Not Returned,1,Rarely
4,TXN500004,CUST003158,2023-10-18,62,Female,Pune,Online,Flipkart,Accessories,Belt,Levis,Red,M,1,2133,3,Standard Day,0,Standard,6,Yes,UPI,3,Not Returned,3,Quarterly


### Inspect column names

In [42]:
print("Dataset columns:\n")

for position, column in enumerate(df.columns, start=1):
    print(f"{position:>2}. {column}")

Dataset columns:

 1. Transaction ID
 2. Customer ID
 3. Purchase Date
 4. Age
 5. Gender
 6. Location
 7. Online/Offline
 8. Online Store
 9. Category
10. Item Purchased
11. Brand
12. Color
13. Size
14. Quantity
15. Purchase Amount (₹)
16. Discount (%)
17. Festival/Sale
18. Shipping Charge (₹)
19. Delivery Speed
20. Delivery Time (Days)
21. Subscription Status
22. Payment Method
23. Review Rating
24. Return Status
25. Previous Purchases
26. Frequency of Purchases


### Dataset information and Data quality

In [43]:
print(df.shape)
df.info()

(10000, 26)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Transaction ID          10000 non-null  str  
 1   Customer ID             10000 non-null  str  
 2   Purchase Date           10000 non-null  str  
 3   Age                     10000 non-null  int64
 4   Gender                  10000 non-null  str  
 5   Location                10000 non-null  str  
 6   Online/Offline          10000 non-null  str  
 7   Online Store            10000 non-null  str  
 8   Category                10000 non-null  str  
 9   Item Purchased          10000 non-null  str  
 10  Brand                   10000 non-null  str  
 11  Color                   10000 non-null  str  
 12  Size                    10000 non-null  str  
 13  Quantity                10000 non-null  int64
 14  Purchase Amount (₹)     10000 non-null  int64
 15  Discount (%)       

#### Data Quality

In [44]:
quality_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique(dropna=False)
})

display(quality_summary)

,Data Type,Missing Values,Missing Percentage,Unique Values
Transaction ID,str,0,0.00,10000
Customer ID,str,0,0.00,3291
Purchase Date,str,0,0.00,730
Age,int64,0,0.00,47
Gender,str,0,0.00,3
Location,str,0,0.00,20
Online/Offline,str,0,0.00,2
Online Store,str,0,0.00,7
Category,str,0,0.00,3
Item Purchased,str,0,0.00,24


### Duplicate and identifier checks

In [45]:
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

print(
    "Duplicate Transaction IDs:",
    df["Transaction ID"].duplicated().sum()
)

print(
    "Unique transactions:",
    df["Transaction ID"].nunique()
)

print(
    "Unique customers:",
    df["Customer ID"].nunique()
)

Exact duplicate rows: 0
Duplicate Transaction IDs: 0
Unique transactions: 10000
Unique customers: 3291


## Inspection Results

- The dataset contains 10,000 transaction records and 26 variables.
- No missing values or exact duplicate rows were identified.
- Every Transaction ID is unique, so it can serve as the primary identifier for each transaction.
- The dataset contains 3,291 unique customers, indicating that some customers made multiple purchases during the observed period.

# Dataset preparation

### Clean text columns

In [46]:
text_columns = df.select_dtypes(include="str").columns

df[text_columns] = df[text_columns].apply(
    lambda column: column.str.strip()
)

print("Leading and trailing spaces removed from text columns.")

Leading and trailing spaces removed from text columns.


### Convert the purchase date

In [47]:
df["Purchase Date"] = pd.to_datetime(
    df["Purchase Date"],
    errors="coerce"
)

print("Invalid purchase dates:", df["Purchase Date"].isna().sum())
print("Earliest purchase:", df["Purchase Date"].min())
print("Latest purchase:", df["Purchase Date"].max())

Invalid purchase dates: 0
Earliest purchase: 2023-01-01 00:00:00
Latest purchase: 2024-12-30 00:00:00


### Validate numerical columns

In [48]:
numeric_columns = [
    "Age",
    "Quantity",
    "Purchase Amount (₹)",
    "Discount (%)",
    "Shipping Charge (₹)",
    "Delivery Time (Days)",
    "Review Rating",
    "Previous Purchases"
]

numeric_validation = df[numeric_columns].agg(
    ["count", "min", "max", "mean", "median"]
).T

display(numeric_validation)

,count,min,max,mean,median
Age,"10,000.00",18.00,64.00,41.10,41.00
Quantity,"10,000.00",1.00,4.00,1.43,1.00
Purchase Amount (₹),"10,000.00",107.00,"113,920.00","6,467.69","3,491.00"
Discount (%),"10,000.00",0.00,69.00,8.70,8.00
Shipping Charge (₹),"10,000.00",0.00,99.00,20.47,0.00
Delivery Time (Days),"10,000.00",0.00,7.00,2.58,2.00
Review Rating,"10,000.00",1.00,5.00,3.48,4.00
Previous Purchases,"10,000.00",0.00,85.00,14.68,9.00


### Inspect Catagorial Data

In [49]:
categorical_columns = [
    "Gender",
    "Online/Offline",
    "Online Store",
    "Category",
    "Festival/Sale",
    "Delivery Speed",
    "Subscription Status",
    "Payment Method",
    "Return Status",
    "Frequency of Purchases"
]

for column in categorical_columns:
    print(f"\n{column}")
    print("-" * len(column))
    print(df[column].value_counts(dropna=False))


Gender
------
Gender
Female    4839
Male      4790
Other      371
Name: count, dtype: int64

Online/Offline
--------------
Online/Offline
Online     7417
Offline    2583
Name: count, dtype: int64

Online Store
------------
Online Store
In-Store Purchase    2583
Meesho               1277
Ajio                 1276
Amazon               1229
Myntra               1218
Flipkart             1212
Nykaa                1205
Name: count, dtype: int64

Category
--------
Category
Clothing       5004
Footwear       3395
Accessories    1601
Name: count, dtype: int64

Festival/Sale
-------------
Festival/Sale
Standard Day                    8835
In-Store Festival Offer          878
Diwali Sale                      124
Independence Day Sale             62
Republic Day Sale                 40
Big Billion Days                  19
Amazon Great Indian Festival      16
Myntra End of Reason Sale         16
Nykaa Pink Friday                 10
Name: count, dtype: int64

Delivery Speed
--------------
Delivery

### Create date features

In [50]:
df["Year"] = df["Purchase Date"].dt.year
df["Month Number"] = df["Purchase Date"].dt.month
df["Month"] = df["Purchase Date"].dt.month_name()
df["Year-Month"] = df["Purchase Date"].dt.to_period("M").astype(str)
df["Quarter"] = "Q" + df["Purchase Date"].dt.quarter.astype("Int64").astype(str)
df["Day of Week"] = df["Purchase Date"].dt.day_name()

### Create age groups

In [51]:
age_bins = [17, 24, 34, 44, 54, 64, np.inf]
age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

df["Age Group"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels
)

### Create analytical flags

In [52]:
df["Returned"] = df["Return Status"].eq("Returned")
df["Subscribed"] = df["Subscription Status"].eq("Yes")
df["Online Purchase"] = df["Online/Offline"].eq("Online")
df["Sale Purchase"] = df["Festival/Sale"].ne("Standard Day")

### Create transaction-level features

In [53]:
df["Average Amount per Item (₹)"] = (
    df["Purchase Amount (₹)"] / df["Quantity"]
)

df["Has Discount"] = df["Discount (%)"].gt(0)
df["Has Shipping Charge"] = df["Shipping Charge (₹)"].gt(0)

### Final validation

In [54]:
print(f"Final dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing values: {df.isna().sum().sum():,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Unique transactions: {df['Transaction ID'].nunique():,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")

display(df.head())

Final dataset shape: 10,000 rows × 40 columns
Missing values: 0


Exact duplicate rows: 0
Unique transactions: 10,000
Unique customers: 3,291


,Transaction ID,Customer ID,Purchase Date,Age,Gender,Location,Online/Offline,Online Store,Category,Item Purchased,Brand,Color,Size,Quantity,Purchase Amount (₹),Discount (%),Festival/Sale,Shipping Charge (₹),Delivery Speed,Delivery Time (Days),Subscription Status,Payment Method,Review Rating,Return Status,Previous Purchases,Frequency of Purchases,Year,Month Number,Month,Year-Month,Quarter,Day of Week,Age Group,Returned,Subscribed,Online Purchase,Sale Purchase,Average Amount per Item (₹),Has Discount,Has Shipping Charge
0,TXN500000,CUST001440,2023-07-22,52,Other,Indore,Offline,In-Store Purchase,Clothing,Jeans,Levis,Black,L,1,2215,6,Standard Day,0,N/A (Offline),0,No,Credit Card,5,Not Returned,2,Quarterly,2023,7,July,2023-07,Q3,Saturday,45-54,False,False,False,False,"2,215.00",True,False
1,TXN500001,CUST002326,2023-01-16,19,Female,Bhubaneswar,Online,Ajio,Clothing,Jacket,Zara,Blue,XS,1,6742,6,Standard Day,0,Express,1,No,Debit Card,4,Returned,3,Quarterly,2023,1,January,2023-01,Q1,Monday,18-24,True,False,True,False,"6,742.00",True,False
2,TXN500002,CUST002680,2023-01-24,40,Male,Bangalore,Online,Myntra,Clothing,Saree,Biba,Red,S,1,4672,0,Standard Day,0,Standard,7,Yes,Cash on Delivery,4,Not Returned,13,Monthly,2023,1,January,2023-01,Q1,Tuesday,35-44,False,True,True,False,"4,672.00",False,False
3,TXN500003,CUST001098,2024-09-06,46,Female,Ahmedabad,Offline,In-Store Purchase,Clothing,Jeans,Levis,White,M,1,1539,5,Standard Day,0,N/A (Offline),0,No,UPI,3,Not Returned,1,Rarely,2024,9,September,2024-09,Q3,Friday,45-54,False,False,False,False,"1,539.00",True,False
4,TXN500004,CUST003158,2023-10-18,62,Female,Pune,Online,Flipkart,Accessories,Belt,Levis,Red,M,1,2133,3,Standard Day,0,Standard,6,Yes,UPI,3,Not Returned,3,Quarterly,2023,10,October,2023-10,Q4,Wednesday,55-64,False,True,True,False,"2,133.00",True,False


### Saving the dataset

In [55]:
OUTPUT_PATH = Path("../data/processed_customer_shopping_behavior.csv")

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Processed dataset saved to:")
# will print the output path to verify that the path is correct and the file has been saved.
#print(OUTPUT_PATH.resolve())

Processed dataset saved to:


# Results 
| Metric | Value |
| :--- | :--- |
| **Raw dataset** | 10,000 rows × 26 columns |
| **Missing values** | 0 |
| **Exact duplicates** | 0 |
| **Unique transactions** | 10,000 |
| **Unique customers** | 3,291 |
| **Date range** | 2023-01-01 to 2024-12-30 |
| **Processed dataset** | 10,000 rows × 40 columns |